# Software Documentation Assistant: Exploration & Indexing
## Week 3 | Retrieval-Augmented Generation Applications (Part C)

This notebook demonstrates the end-to-end ingestion and indexing pipeline for developer technical documentation:
1. Document loading with YAML frontmatter parsing
2. Code-aware hierarchical chunking preserving Markdown code blocks
3. Dense vector embeddings with SentenceTransformers (`all-MiniLM-L6-v2`)
4. Sparse BM25 lexical indexing
5. Index persistence and verification

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

### 1. Ingest Technical Documents
We load our curated technical documentation corpus across FastAPI (v0.115 and legacy v0.90) and Pydantic (v2.0 and v1.10).

In [ ]:
from src.document_loader import DocumentLoader
from src.config import settings

loader = DocumentLoader(settings.DOCS_DIR)
docs = loader.load_all()
print(f"Loaded {len(docs)} technical documentation files:\n")
for d in docs:
    print(f"- [{d.doc_id}] {d.title} (Framework: {d.framework}, Ver: {d.version}, Chars: {d.char_count})")

### 2. Code-Aware Hierarchical Chunking
Notice that code blocks (` ```python ... ``` `) are kept intact to preserve syntactic validity.

In [ ]:
from src.chunker import MarkdownCodeAwareChunker

chunker = MarkdownCodeAwareChunker(target_chunk_size=settings.CHUNK_SIZE, chunk_overlap=settings.CHUNK_OVERLAP)
chunks = chunker.chunk_all(docs)
code_chunks = [c for c in chunks if c.has_code]

print(f"Total chunks generated: {len(chunks)}")
print(f"Chunks containing code examples: {len(code_chunks)} ({len(code_chunks)/len(chunks)*100:.1f}%)")

# Inspect a sample code chunk
sample = code_chunks[0]
print(f"\nSample Chunk ID: {sample.chunk_id}")
print(f"Citation: {sample.citation}")
print("Content Preview:")
print(sample.content[:300] + "...")

### 3. Build Hybrid Vector Store
Compute normalized dense vectors and construct the BM25 lexical inverted index.

In [ ]:
from src.rag_pipeline import RAGPipeline

pipeline = RAGPipeline()
index_stats = pipeline.build_or_load_index(force_reindex=False)
print("Index status:", index_stats)
print("Pipeline stats:", pipeline.get_stats())